In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
profit=pd.read_csv('ecommerce_sales.csv')
profit['returned']=profit['returned'].map({'Yes':1,'No':0})
profit = profit.drop(['order_id', 'customer_id', 'product_id', 'order_date'], axis=1, errors='ignore')
numerical_col=profit.select_dtypes(include=['int64','float64'])
categorical_col=profit.select_dtypes(include=['object'])
profit_encoded = pd.get_dummies(profit, drop_first=True)

In [3]:
target='profit_margin'
important_col=['price','quantity','total_amount','shipping_cost']
important_col2=numerical_col

In [4]:
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error,confusion_matrix,classification_report
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split,GridSearchCV

In [5]:
models = {
    'SVR': SVR(),
    'decision_tree': DecisionTreeRegressor(random_state=42),
    'random_forest': RandomForestRegressor(random_state=42),
    'XGB': XGBRegressor(random_state=42),
    'LGBM': LGBMRegressor(random_state=42, verbose=-1)
}

results = []
X = profit[important_col]
y = profit[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    
    results.append({
        'Model': name,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2 Score': r2
    })

results_df = pd.DataFrame(results).sort_values(by='R2 Score', ascending=False)
print(results_df)

           Model        MAE         MSE       RMSE  R2 Score
2  random_forest   9.871602  476.893122  21.837883  0.812159
4           LGBM   9.889836  482.148420  21.957878  0.810089
3            XGB   9.922072  527.322534  22.963504  0.792296
0            SVR   9.629499  776.416521  27.864252  0.694181
1  decision_tree  11.477093  896.110719  29.935108  0.647036


In [ ]:
param_grid = {
    'n_estimators': [100, 200,300],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']    
}
rf=RandomForestRegressor(random_state=42)

grid_search=GridSearchCV(estimator=rf,param_grid=param_grid,cv=5,scoring='r2',n_jobs=-1)
grid_search.fit(X_train,y_train)

print(f"best hyperparameters :{grid_search.best_params_}")
print(f"best training r2_score:{grid_search.best_score_:.3f}")

best_rf = grid_search.best_estimator_
y_pred_rf = best_rf.predict(X_test)
final_test_r2 = r2_score(y_test, y_pred_rf)

print(f"Final Optimized Test R2 Score: {final_test_r2:.4f}")

In [ ]:
importances = best_rf.feature_importances_

feature_imp_df = pd.DataFrame({
    'Feature': important_col,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(data=feature_imp_df, x='Importance', y='Feature', palette='viridis')
plt.title('profit ')
plt.xlabel('profits ')
plt.ylabel('Features')
plt.tight_layout()
plt.show()


residuals = y_test - y_pred_rf


plt.figure(figsize=(8, 5))
sns.histplot(residuals, kde=True, bins=30)
plt.axvline(0, color='red', linestyle='--')
plt.title('Residuals Distribution ')
plt.xlabel('Error ')
plt.show()